# LoRA 推理 - Google Colab 版

加载微调后的模型进行角色扮演对话

**运行环境**: Google Colab (CPU 或 GPU)

## 1. 环境准备

In [ ]:
# 安装依赖
!pip install -q transformers peft bitsandbytes accelerate safetensors

In [ ]:
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

## 2. 上传模型文件

In [ ]:
# 方法一: 从 Google Drive 加载
# from google.colab import drive
# drive.mount('/content/gdrive')
# MODEL_PATH = "/content/gdrive/MyDrive/your-model-path"

# 方法二: 上传 zip 文件并解压
from google.colab import files
import os

# 如果有训练好的模型，上传并解压
# uploaded = files.upload()
# !unzip -q lora_model.zip -d ./output/lora_roleplay/final_model

# 或直接指定 HuggingFace 模型路径（首次运行会下载）
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH = "./output/lora_roleplay/final_model"  # 如果有本地 adapter

os.makedirs("outputs", exist_ok=True)

## 3. 加载模型

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

print("=== 加载分词器 ===")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("=== 加载基座模型 ===")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    dtype=torch.float16,
    device_map="auto",
)

# 加载 LoRA adapter（如果有）
import os
if os.path.exists(ADAPTER_PATH):
    print(f"=== 加载 LoRA adapter: {ADAPTER_PATH} ===")
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, dtype=torch.float16)
else:
    print("未找到 LoRA adapter，使用基座模型")
    model = base_model

model.eval()
print("模型加载完成")

## 4. 定义角色卡和对话函数

In [ ]:
import json

# 内置角色卡示例
CHARACTER_CARDS = {
    "alina": {
        "name": "Alina",
        "persona": "A mysterious girl with magical powers, calm and wise.",
        "background": "She comes from a distant land where magic flows freely.",
        "speech_style": "Speaks gently and often uses metaphors."
    },
    "luoji": {
        "name": "罗辑",
        "persona": "面壁者，冷静理智，具有超强的逻辑思维能力。",
        "background": "《三体》系列主角，曾作为人类文明的执剑人。",
        "speech_style": "言简意赅，富有哲理。"
    }
}

def build_character_desc(card):
    """构建角色描述文本"""
    parts = []
    for key in ["name", "persona", "background", "speech_style"]:
        if key in card:
            parts.append(f"{key}: {card[key]}")
    return "\n".join(parts)

def chat(model, tokenizer, character_card: str, user_input: str,
         max_new_tokens: int = 512, temperature: float = 0.7) -> str:
    """生成角色扮演回复"""
    messages = [
        {"role": "system", "content": character_card},
        {"role": "user", "content": user_input}
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "assistant" in response:
        return response.split("assistant")[-1].strip()
    return response[len(input_text):].strip()

## 5. 交互式对话

In [ ]:
# 选择角色
selected_character = "alina"  # 可选: "alina", "luoji"
character_card = build_character_desc(CHARACTER_CARDS[selected_character])
print(f"【角色】{CHARACTER_CARDS[selected_character]['name']}")
print(f"【人设】{CHARACTER_CARDS[selected_character]['persona']}")
print("-" * 50)

# 开始对话
print("开始对话（输入 exit 退出）:")
while True:
    user_input = input("\n【用户】: ").strip()
    if user_input.lower() in ["exit", "quit", "q"]:
        print("对话结束")
        break
    if not user_input:
        continue
    
    response = chat(model, tokenizer, character_card, user_input)
    print(f"\n【角色】: {response}")

## 6. 批量测试

In [ ]:
# 批量测试输入
test_inputs = [
    "你好，请介绍一下你自己。",
    "你能告诉我一些关于这个世界的事情吗？",
    "你有什么特殊的能力吗？",
]

print("=" * 50)
print("批量测试")
print("=" * 50)

for i, user_input in enumerate(test_inputs, 1):
    print(f"\n【测试 {i}】")
    print(f"【用户】: {user_input}")
    response = chat(model, tokenizer, character_card, user_input)
    print(f"【角色】: {response}")
    print("-" * 50)